# Week 02: Mini Semantic Search

Search FAQs by meaning using embeddings and ChromaDB.

In [ ]:
import numpy as np
import pandas as pd
import chromadb
from sentence_transformers import SentenceTransformer

## Step 1: Load data

In [ ]:
df = pd.read_csv('data/faqs.csv')
print(f'Loaded {len(df)} FAQs')
df.head()

## Step 2: Clean data

In [ ]:
before = len(df)
df = df.drop_duplicates(subset=['question'])
df = df.dropna(subset=['question','answer'])
df['question'] = df['question'].str.strip()
df['answer'] = df['answer'].str.strip()
print(f'Cleaned: {before} -> {len(df)}')

## Step 3: Chunk text

In [ ]:
df['chunk'] = df['question'] + ' ' + df['answer']
chunks = df['chunk'].tolist()
print(f'Created {len(chunks)} chunks')

## Step 4: Embed

In [ ]:
model = SentenceTransformer('all-MiniLM-L6-v2')
embeddings = model.encode(chunks)
print(f'Shape: {embeddings.shape}')

## Step 5: Store in ChromaDB

In [ ]:
client = chromadb.Client()
collection = client.create_collection('faqs')
collection.add(ids=[str(i) for i in range(len(chunks))], embeddings=embeddings.tolist(), documents=chunks, metadatas=[{'question':q,'answer':a} for q,a in zip(df['question'],df['answer'])])
print(f'Stored {collection.count()}')

## Step 6: Search function

In [ ]:
def search(query, n=3):
    qe = model.encode([query]).tolist()
    return collection.query(query_embeddings=qe, n_results=n)

## Step 7: Test queries

In [ ]:
for q in ['How do I track changes in my code?', 'What tool helps apps run everywhere?', 'How can I find patterns in data?']:
    print(f'\nQuery: {q}')
    r = search(q)
    for i, m in enumerate(r['metadatas'][0]):
        print(f'  {i+1}. {m["question"]}')

## Step 8: NumPy stats

In [ ]:
df['q_length'] = df['question'].str.len()
lens = df['q_length'].to_numpy()
print(f'Avg: {np.mean(lens):.1f}')
print(f'Median: {np.median(lens):.1f}')
print(f'Max: {np.max(lens)}')
print(f'75th: {np.percentile(lens, 75):.1f}')

## Summary

Loaded 15 FAQs, cleaned, chunked, embedded with all-MiniLM-L6-v2, stored in ChromaDB. Search matches by meaning.